# Phase 4 and 5 — Data Cleaning and Text Preprocessing

## Purpose

This notebook is executed only after completing `02_data_understanding.ipynb`.

It applies the cleaning decisions to the frozen raw dataset and exports the
official analysis-ready dataset.

The original raw file is never overwritten.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Professional project path:
RAW_PATH = Path("../data/raw/combined_reddit_posts.jsonl")
OUTPUT_DIR = Path("../data/processed")
SCRIPT_PATH = Path("../src/prepare_analysis_dataset.py")

# Fallbacks make the uploaded notebook runnable in the current environment.
if not RAW_PATH.exists():
    RAW_PATH = Path("/mnt/data/combined_reddit_posts.jsonl")
if not SCRIPT_PATH.exists():
    SCRIPT_PATH = Path("/mnt/data/prepare_analysis_dataset.py")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SCRIPT_PATH.parent))

from prepare_analysis_dataset import prepare_dataset, write_jsonl

print("Raw dataset:", RAW_PATH)
print("Reusable cleaning module:", SCRIPT_PATH)


## 1. Load and normalize

Different sources use different identifiers, date fields, and text fields. The reusable module:

1. coalesces `id`, `post_id`, and `sample_id`;
2. creates a source-qualified `record_id`;
3. generates a deterministic text-hash ID where no source ID exists;
4. standardizes platform and timestamp fields;
5. preserves all source-specific columns.


In [ ]:
full_df, ready_df, excluded_df, summary_df = prepare_dataset(RAW_PATH)

display(summary_df)
print("\nRaw shape:", full_df.shape)
print("Analysis-ready shape:", ready_df.shape)


## 2. Source and platform balance

Source imbalance matters because a model trained or summarized on the full dataset may mostly describe the largest sources rather than developers generally. Always report results by platform before presenting an overall total.


In [ ]:
platform_profile = (
    full_df.groupby("platform")
    .agg(
        raw_rows=("record_id", "size"),
        ai_relevant_rows=("is_ai_relevant", "sum"),
        analysis_ready_rows=("analysis_eligible", "sum"),
        missing_dates=("date_known", lambda s: (~s).sum()),
    )
    .sort_values("raw_rows", ascending=False)
)
platform_profile["strict_relevance_rate"] = (
    platform_profile["ai_relevant_rows"] / platform_profile["raw_rows"]
)
display(platform_profile)


## 3. Why the original relevance rule needs correction

The collectors used substring matching such as:

```python
"ai" in text.lower()
```

This can match unrelated words. The repaired rule uses:

```python
r"(?<![A-Za-z0-9_])ai(?![A-Za-z0-9_])"
```

It also recognizes named tools such as ChatGPT, Claude, Gemini, Copilot, Cursor, DeepSeek, LLM, Llama, Mistral, and Qwen.


In [ ]:
source_relevance = (
    full_df.groupby("source")
    .agg(
        rows=("record_id", "size"),
        strict_ai_rows=("is_ai_relevant", "sum"),
    )
)
source_relevance["strict_ai_rate"] = (
    source_relevance["strict_ai_rows"] / source_relevance["rows"]
)
display(source_relevance.sort_values("strict_ai_rate"))


## 4. Duplicate and date checks

- IDs are namespaced by source to prevent cross-platform collisions.
- Exact duplicate text is detected after whitespace and markup normalization.
- Missing dates are retained but excluded from timeline conclusions.
- Records before 2025 are flagged because the research question emphasizes recent discussions.


In [ ]:
quality_checks = {
    "Unique record IDs": ready_df["record_id"].is_unique,
    "No blank analysis text": ready_df["text_clean_basic"].str.strip().ne("").all(),
    "No exact-text duplicates": ~ready_df["text_hash"].duplicated().any(),
    "All rows pass strict AI relevance": ready_df["is_ai_relevant"].all(),
    "All rows have at least five words": ready_df["word_count"].ge(5).all(),
}
display(pd.Series(quality_checks, name="passed"))

print("\nDate coverage in analysis-ready data:")
display(
    ready_df[["date_known", "is_recent_2025_plus"]]
    .value_counts(dropna=False)
    .rename("rows")
)


## 5. Research-theme coverage

These flags do **not** replace sentiment or emotion models. They are transparent keyword indicators used to understand whether the corpus actually contains enough material about stress, anxiety, burnout, job security, productivity, trust, and privacy.


In [ ]:
theme_counts = (
    ready_df["research_themes"]
    .explode()
    .dropna()
    .value_counts()
    .rename_axis("theme")
    .reset_index(name="rows")
)
display(theme_counts)


## 6. Export

Use `analysis_ready_posts.jsonl` for EDA and modeling. Keep `excluded_low_relevance_posts.jsonl` as an audit trail instead of deleting questionable records permanently.


In [ ]:
READY_PATH = OUTPUT_DIR / "analysis_ready_posts.jsonl"
EXCLUDED_PATH = OUTPUT_DIR / "excluded_low_relevance_posts.jsonl"
SUMMARY_PATH = OUTPUT_DIR / "data_quality_summary.csv"

write_jsonl(ready_df, READY_PATH)
write_jsonl(excluded_df, EXCLUDED_PATH)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved:", READY_PATH)
print("Saved:", EXCLUDED_PATH)
print("Saved:", SUMMARY_PATH)


## Modeling rule

Do not use one destructively cleaned column for every method:

- **VADER:** use `text_clean_basic` because punctuation, capitalization, emojis, and negation carry sentiment.
- **Transformer sentiment model:** use `text_clean_basic`; do not remove stopwords or lemmatize.
- **TF-IDF / LDA:** begin with `text_clean_lexical`, then apply a task-specific tokenizer and stopword list.
- **BERTopic:** use `text_clean_basic` or lightly cleaned raw text because sentence embeddings need natural language context.

The source-provided `sentiment_label` and `prediction` fields should remain reference columns. Store your own outputs under new names such as `vader_label` and `transformer_label`.
